# تست دود Colab برای پروژهٔ واقعی شما: `perfect_project` (۲۴۶ نماد کاراکتری)

این نوت‌بوک مخصوص پروژهٔ `perfect_project` (شاخهٔ `fix/fa-gpu-readiness`) است که توکنایزر **کاراکتری ۲۴۶ نمادی** و `n_vocab=246` دارد.

> **هشدار مهم:** کد شاخهٔ `codex` روی GitHub (espeak/۱۷۸) را اجرا **نکنید** — با دادهٔ ۲۴۶تایی و چک‌پوینت‌های شما ناسازگار است و کرش می‌کند. حتماً کد خودِ `perfect_project` را بیاورید (سلول ۳).

سلول‌ها را به‌ترتیب اجرا کنید. خروجی هر مرحله روی Google Drive ذخیره می‌شود.

### پیش‌نیاز
- Runtime → Change runtime type → **GPU**.
- لینک‌های Drive روی «هرکسی با لینک» (Anyone with the link) باز باشد، وگرنه `gdown` خطا می‌دهد.
- کد `perfect_project` را یا به‌صورت **zip در Drive** بگذارید (و `CODE_ZIP_ID` را پر کنید)، یا در یک **repo گیت** و `CODE_GIT_URL` را پر کنید.

In [ ]:
# [1] بررسی GPU
!nvidia-smi

In [ ]:
# [2] اتصال Drive + پوشهٔ خروجی
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_OUT = "/content/drive/MyDrive/zero_tts_run"   # در صورت نیاز تغییر دهید
os.makedirs(DRIVE_OUT, exist_ok=True)
print("outputs ->", DRIVE_OUT)

In [ ]:
# [3] آوردن کد perfect_project به Colab  (یکی از دو روش را پر کنید)
#  روش A: zip کد را در Drive گذاشته‌اید -> file id آن را اینجا بگذارید
CODE_ZIP_ID = ""          # مثال: "1AbC...xyz"  (zip کل پوشهٔ perfect_project)
#  روش B: کد را در یک repo گیت دارید
CODE_GIT_URL = ""         # مثال: "https://github.com/<you>/perfect_project.git"
CODE_GIT_BRANCH = "fix/fa-gpu-readiness"

%cd /content
!rm -rf perfect_project
!pip -q install gdown

if CODE_ZIP_ID:
    !gdown "https://drive.google.com/uc?id={CODE_ZIP_ID}" -O /content/code.zip
    !mkdir -p /content/perfect_project && unzip -o -q /content/code.zip -d /content/perfect_project
    # اگر zip یک پوشهٔ ریشه داشت، آن را پیدا کن:
    import glob, os
    sub = [p for p in glob.glob('/content/perfect_project/*') if os.path.isdir(p)]
    if len(sub) == 1 and not os.path.exists('/content/perfect_project/train_ttv_v1.py'):
        inner = sub[0]
        !shopt -s dotglob; mv "{inner}"/* /content/perfect_project/ 2>/dev/null; true
elif CODE_GIT_URL:
    !git clone "{CODE_GIT_URL}" perfect_project
    %cd /content/perfect_project
    !git checkout "{CODE_GIT_BRANCH}" || echo "branch not found, staying on default"
    %cd /content
else:
    raise SystemExit("یکی از CODE_ZIP_ID یا CODE_GIT_URL را پر کنید.")

%cd /content/perfect_project
print("\n--- محتوای ریشهٔ پروژه ---")
!ls -1 | head -40
assert os.path.exists("train_ttv_v1.py"), "train_ttv_v1.py پیدا نشد؛ ساختار zip را بررسی کنید."

In [ ]:
# [4] نصب وابستگی‌ها + build monotonic_align
%cd /content/perfect_project
!apt-get -qq update && apt-get -qq install -y espeak-ng
!pip -q install -r requirements.txt || echo "requirements ممکن است بخشی نصب نشود؛ ادامه می‌دهیم"
!pip -q install hazm gdown
# build cython (نسخهٔ .so داخل مخزن برای پایتون قدیمی است):
import os
if os.path.isdir("ttv_v1/monotonic_align"):
    %cd /content/perfect_project/ttv_v1/monotonic_align
    !python setup.py build_ext --inplace
    %cd /content/perfect_project
# تا زیرفرایندها monotonic_align را پیدا کنند:
os.environ["PYTHONPATH"] = "/content/perfect_project:/content/perfect_project/ttv_v1:" + os.environ.get("PYTHONPATH", "")
print("PYTHONPATH =", os.environ["PYTHONPATH"])

In [ ]:
# [5] preflight: تأیید ۲۴۶ نماد + تست CPU dry-run
%cd /content/perfect_project
!python -c "from ttv_v1.text.symbols import symbols; print('symbols:', len(symbols), '| index0:', repr(symbols[0]))"
!espeak-ng -v fa "سلام" --ipa || echo "(espeak اختیاری برای این پروژه)"
!python tests/test_cpu_dry_run.py || echo "اگر این‌جا خطا داد، قبل از ادامه باید رفع شود"

In [ ]:
# [6] دانلود چک‌پوینت‌ها و داده‌ها از Drive (شناسه‌های شما)
import os
os.makedirs("/content/ckpts", exist_ok=True)
os.makedirs("/content/data_zips", exist_ok=True)

CKPTS = {
    "G_3135000.zip": "1mUDS-Zh8m8P_p6REeRUDjkn7bMDSjkPM",   # مدل فارسی قدیمی (zip)
    "G_0.pth":       "187G1lPWLa__wSeGuLrNP1IlcOXkWholc",   # چک‌پوینت پایه
}
DATA_IDS = [
    "1zA_CO5_yao5f9Gg2OwLxwdTFxHxM2YuP",
    "15vnhZ2a_dnymdYYHJ7AQktb18d-yzJ0F",
    "13gVShTSC-OM1eHeuJRjCFinHC_1LIKpj",
    "1vfRUEIO_OgZG9nij-CfCRtMTK2D__qKP",
    "1uW82gIKugERwHnynBQKNK_xmp8ZU_sm5",
    "14M8BD52Nw22imoKG_9v2atLevJun5DAv",
    "1BQT5j3Zb8tQT140aVRzzgnYmdq0UGI8L",
]

for name, fid in CKPTS.items():
    !gdown "https://drive.google.com/uc?id={fid}" -O /content/ckpts/{name}
%cd /content/data_zips
for fid in DATA_IDS:
    !gdown "https://drive.google.com/uc?id={fid}"   # با نام اصلی ذخیره می‌شود
%cd /content/perfect_project
!ls -lh /content/ckpts /content/data_zips

In [ ]:
# [7] تشخیص نوع فایل‌ها + باز کردن zipها + دیدن ساختار داده
import glob
print("== نوع فایل‌ها ==")
for f in sorted(glob.glob("/content/data_zips/*")) + sorted(glob.glob("/content/ckpts/*")):
    !file "{f}"
!mkdir -p /content/ckpts_x /content/data
for f in glob.glob("/content/ckpts/*.zip"):
    !unzip -o -q "{f}" -d /content/ckpts_x
for f in glob.glob("/content/data_zips/*.zip"):
    !unzip -o -q "{f}" -d /content/data
print("\n== چک‌پوینت‌های استخراج‌شده ==")
!find /content/ckpts_x -maxdepth 3 -name "*.pth"
print("\n== ساختار داده ==")
!find /content/data -maxdepth 4 | head -80

In [ ]:
# [8] بازرسی واژگانِ چک‌پوینت‌ها — مهم‌ترین سلول تصمیم
# مدل perfect_project با n_vocab=246 فقط چک‌پوینتی را resume می‌کند که emb آن (246, ...) باشد.
import torch, glob
cands = sorted(glob.glob("/content/ckpts_x/**/G_*.pth", recursive=True)) + sorted(glob.glob("/content/ckpts/*.pth"))
print("چک‌پوینت‌های یافت‌شده:", cands, "\n")
assert cands, "هیچ G_*.pth پیدا نشد."
for ck in cands:
    try:
        d = torch.load(ck, map_location="cpu")
    except Exception as e:
        print(ck, "-> خطای لود:", e); continue
    sd = d["model"] if isinstance(d, dict) and "model" in d else d
    emb = next((tuple(v.shape) for k, v in sd.items() if k.endswith("enc_p.emb.weight")), None)
    cls = next((tuple(v.shape) for k, v in sd.items() if "phoneme_classifier" in k), None)
    it  = d.get("iteration") if isinstance(d, dict) else None
    print(f"{ck}\n  enc_p.emb.weight={emb}  phoneme_classifier={cls}  iteration={it}")
    if emb:
        if emb[0] == 246:
            print("  => ✅ سازگار با ۲۴۶: مستقیماً قابل ادامه (سلول ۱۱ این را کپی می‌کند).")
        else:
            print(f"  => ⚠️ n_vocab={emb[0]} (≠۲۴۶): برای استفاده، transfer_checkpoint.py لازم است (warm-start).")
    print()

### تنظیم مسیر داده + تصمیم چک‌پوینت

از خروجی سلول ۷ ساختار واقعی داده را ببینید و در سلول ۱۰ مسیر `datasets/` را مطابق آن قرار دهید. ساختار موردنیاز `prepare_filelist.py`:

```
datasets/<speaker>/wavs/001.wav   (16kHz mono)
datasets/<speaker>/wavs/001.txt   (یا texts/001.txt) متن فارسی همان جمله
```

از خروجی سلول ۸:
- اگر `G_0.pth` یا هر چک‌پوینتی `emb=(246,...)` بود → همان را در سلول ۱۱ به‌عنوان مبنا کپی کنید (ادامهٔ آموزش).
- اگر فقط `(178,...)` داشتید → یا از صفر شروع کنید، یا اول `python transfer_checkpoint.py` را (با مسیرهای درست) اجرا کنید تا warm-start ۲۴۶ بسازد.

In [ ]:
# [10] preprocess با اسکریپت‌های خودِ perfect_project
# مسیر datasets را مطابق خروجی سلول ۷ تنظیم کنید (یا داده‌های استخراج‌شده را به datasets/ لینک کنید):
%cd /content/perfect_project
import os
os.environ["PYTHONPATH"] = "/content/perfect_project:/content/perfect_project/ttv_v1:" + os.environ.get("PYTHONPATH","")

# اگر دادهٔ شما جای دیگری باز شده، یک symlink بسازید:
# !rm -rf datasets && ln -s /content/data/<your_dataset_root> datasets

# 1) filelist (با فیلتر CTC و max_text_len)
!python ttv_v1/preprocessing/prepare_filelist.py --datasets_dir datasets --config config_cpu_smoke.json
# 2) اعتبارسنجی
!python scripts/validate_dataset.py || true
# 3) توکن (کاراکتری ۲۴۶ از config)
!python ttv_v1/preprocessing/extract_token.py --config config_cpu_smoke.json --input_filelist filelists/train_wav.txt
# 4) F0 (pyin، Hz خام)
!python ttv_v1/preprocessing/extract_f0.py --input_filelist filelists/train_wav.txt --skip_existing
# 5) W2V (روی GPU)
!python ttv_v1/preprocessing/extract_w2v.py --input_filelist filelists/train_wav.txt --device cuda --skip_existing
# 6) گیت نهایی
!python pipeline_check.py --full-report || true
print("--- filelists ---"); !wc -l filelists/train_*.txt 2>/dev/null; !head -n 2 filelists/train_wav.txt

In [ ]:
# [11] (اختیاری) قرار دادن چک‌پوینت مبنا برای ادامهٔ آموزش
# فقط اگر سلول ۸ نشان داد چک‌پوینتی emb=(246,...) دارد.
import glob, shutil, os
os.makedirs("checkpoints/colab_smoke", exist_ok=True)
SRC = "/content/ckpts/G_0.pth"   # یا مسیر چک‌پوینت ۲۴۶ سازگار از سلول ۸
if os.path.exists(SRC):
    shutil.copy(SRC, "checkpoints/colab_smoke/G_0.pth")
    print("مبنا برای resume کپی شد:", SRC)
else:
    print("چک‌پوینت مبنا کپی نشد → از صفر آموزش می‌بیند (برای تست دود قابل قبول است).")

In [ ]:
# [12] آموزش تست دود (config_cpu_smoke: 3 epoch, batch 2, fp16=false)
%cd /content/perfect_project
import os
os.environ["PYTHONPATH"] = "/content/perfect_project:/content/perfect_project/ttv_v1:" + os.environ.get("PYTHONPATH","")
!CUDA_VISIBLE_DEVICES=0 python train_ttv_v1.py -c config_cpu_smoke.json -m checkpoints/colab_smoke
# معیار موفقیت: بدون crash، loss متناهی (نه NaN)، و ساخته‌شدن checkpoints/colab_smoke/G_*.pth

In [ ]:
# [13] ذخیرهٔ خروجی‌ها روی Google Drive
!mkdir -p "{DRIVE_OUT}/colab_smoke" "{DRIVE_OUT}/filelists"
!rsync -a checkpoints/colab_smoke "{DRIVE_OUT}/colab_smoke/" 2>/dev/null || cp -r checkpoints/colab_smoke "{DRIVE_OUT}/colab_smoke/"
!cp -r filelists "{DRIVE_OUT}/filelists/" 2>/dev/null || true
!cp config_cpu_smoke.json "{DRIVE_OUT}/" 2>/dev/null || true
print("ذخیره شد در:", DRIVE_OUT)
!ls -lh "{DRIVE_OUT}/colab_smoke"

## بعد از تست دود موفق

اگر سلول ۱۲ بدون خطا اجرا شد، `loss` متناهی بود و `checkpoints/colab_smoke/G_*.pth` ساخته شد → مسیر سالم است.

### آموزش کامل GPU (~۸ روز)
۱. **کل** داده‌ها را preprocess کنید (`bash run_preprocessing.sh`).
۲. چک‌پوینت مبنای ۲۴۶ را در `checkpoints/persian_main/` بگذارید (G_0 سازگار یا خروجی `transfer_checkpoint.py` / `init_finetune_from_pretrained.py`).
۳. در `tmux`/`screen`:  `bash run_gpu_training.sh start`  (یا `python train_ttv_v1.py -c config_gpu_3090.json -m checkpoints/persian_main`).
۴. هر چند ساعت `checkpoints/` را با `rsync` روی Drive بک‌آپ بگیرید.
۵. resume بعد از قطعی: همان دستور را دوباره بزنید (آخرین `G_*.pth` خودکار لود می‌شود).

> راهنمای کامل: `docs/RAHNAMA_NAHAYI_FA.md`